# Tuned LightGBM + CatBoost + XGBoost Stack

Pipeline: Feature engineering → importance-based feature selection → Bayesian-tuned base models → seed-averaged predictions → Ridge/Lasso meta-learner.

Changes: 
1. `log1p(y_1)` target transform (right-skewed, confirmed by EDA)
2. Feature selection — drop noise features via LightGBM importance
3. Dropped KNN and ElasticNet from the stack — tree-only now
4. Seed averaging (3 seeds per model) for variance reduction
5. All 3 models Optuna-tuned per target

Validation: GroupKFold by patient `sub_id`, no leakage.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
SEEDS = [42, 123, 7]
np.random.seed(SEED)
N_FOLDS = 5

## 1. Load Data & EDA

In [10]:
df_train = pd.read_csv("src_files/train_competition_2026.csv")
df_test = pd.read_csv("src_files/test_no_outcome.csv")

print(f"Raw train: {df_train.shape}")
print(f"Raw test:  {df_test.shape}")

Raw train: (432600, 18)
Raw test:  (103500, 16)


Looking at the target distributions to decide if we need any transformations (e.g., `log1p` for skewed targets).

In [ ]:
y_train_eda = df_train.groupby("obs")[["y_1", "y_2"]].first()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

axes[0, 0].hist(y_train_eda["y_1"], bins=60, edgecolor="black", alpha=0.7, color="steelblue")
axes[0, 0].set_title("y_1 — Raw Distribution")
axes[0, 0].set_xlabel("y_1")
axes[0, 0].axvline(y_train_eda["y_1"].median(), color="red", ls="--", label=f'median={y_train_eda["y_1"].median():.1f}')
axes[0, 0].axvline(y_train_eda["y_1"].mean(), color="orange", ls="--", label=f'mean={y_train_eda["y_1"].mean():.1f}')
axes[0, 0].legend()

y1_log = np.log1p(y_train_eda["y_1"])
axes[0, 1].hist(y1_log, bins=60, edgecolor="black", alpha=0.7, color="forestgreen")
axes[0, 1].set_title("y_1 — log1p(y_1) Distribution")
axes[0, 1].set_xlabel("log1p(y_1)")
axes[0, 1].axvline(y1_log.median(), color="red", ls="--", label=f"median={y1_log.median():.2f}")
axes[0, 1].axvline(y1_log.mean(), color="orange", ls="--", label=f"mean={y1_log.mean():.2f}")
axes[0, 1].legend()

from scipy import stats
axes[0, 2].set_title("y_1 — QQ Plot (raw vs log1p)")
stats.probplot(y_train_eda["y_1"], dist="norm", plot=axes[0, 2])
axes[0, 2].get_lines()[0].set_color("steelblue")
axes[0, 2].get_lines()[0].set_label("raw")

axes[1, 0].hist(y_train_eda["y_2"], bins=60, edgecolor="black", alpha=0.7, color="steelblue")
axes[1, 0].set_title("y_2 — Raw Distribution")
axes[1, 0].set_xlabel("y_2")
axes[1, 0].axvline(y_train_eda["y_2"].median(), color="red", ls="--", label=f'median={y_train_eda["y_2"].median():.1f}')
axes[1, 0].axvline(y_train_eda["y_2"].mean(), color="orange", ls="--", label=f'mean={y_train_eda["y_2"].mean():.1f}')
axes[1, 0].legend()

y2_log = np.log1p(y_train_eda["y_2"])
axes[1, 1].hist(y2_log, bins=60, edgecolor="black", alpha=0.7, color="forestgreen")
axes[1, 1].set_title("y_2 — log1p(y_2) Distribution")
axes[1, 1].set_xlabel("log1p(y_2)")
axes[1, 1].axvline(y2_log.median(), color="red", ls="--", label=f"median={y2_log.median():.2f}")
axes[1, 1].axvline(y2_log.mean(), color="orange", ls="--", label=f"mean={y2_log.mean():.2f}")
axes[1, 1].legend()

axes[1, 2].set_title("y_2 — QQ Plot (raw)")
stats.probplot(y_train_eda["y_2"], dist="norm", plot=axes[1, 2])
axes[1, 2].get_lines()[0].set_color("steelblue")

plt.tight_layout()
plt.show()

from scipy.stats import skew, kurtosis
print("Target distribution summary:")
for col in ["y_1", "y_2"]:
    vals = y_train_eda[col]
    print(f"\n  {col}:")
    print(f"    range: [{vals.min():.1f}, {vals.max():.1f}], mean: {vals.mean():.1f}, median: {vals.median():.1f}")
    print(f"    std: {vals.std():.1f}, skew: {skew(vals):.3f}, kurtosis: {kurtosis(vals):.3f}")
    print(f"    log1p skew: {skew(np.log1p(vals)):.3f}")

print("\n=> y_1 is right-skewed — log1p transform will help.")
print("=> y_2 is roughly symmetric — no transform needed.")

## 2. Feature Engineering

Each observation is a ~30-row time series block. We collapse it into a single feature vector per observation using aggregation stats, temporal features, FFT, energy, cross-channel interactions, etc.

In [ ]:
T_COLS = [c for c in df_train.columns if c.startswith("t_")]
NUM_COLS = [c for c in df_train.columns if c.startswith("num_")]
CAT_COLS = [c for c in df_train.columns if c.startswith("cat_")]
N_T = len(T_COLS)


def collapse_obs(df):
    obs_groups = df.groupby("obs")
    unique_obs = np.sort(df["obs"].unique())
    obs_idx = pd.Index(unique_obs, name="obs")
    n_obs = len(unique_obs)

    static = obs_groups[NUM_COLS + CAT_COLS].first()

    basic = obs_groups[T_COLS].agg(["mean", "std", "min", "max", "median", "first", "last"])
    basic.columns = [f"{f}_{a}" for f, a in basic.columns]

    quants = {}
    for q, label in [(0.10, "q10"), (0.25, "q25"), (0.75, "q75"), (0.90, "q90")]:
        qdf = obs_groups[T_COLS].quantile(q)
        qdf.columns = [f"{c}_{label}" for c in T_COLS]
        quants[label] = qdf

    iqr = quants["q75"].values - quants["q25"].values
    iqr_df = pd.DataFrame(iqr, index=obs_idx, columns=[f"{c}_iqr" for c in T_COLS])

    obs_size = obs_groups.size()
    max_len = obs_size.max()
    t_data = df[T_COLS].values

    padded = np.full((n_obs, max_len, N_T), np.nan)
    lengths = np.zeros(n_obs, dtype=int)
    row = 0
    for i in range(n_obs):
        n = obs_size.iloc[i]
        padded[i, :n, :] = t_data[row:row + n, :]
        lengths[i] = n
        row += n

    from scipy.stats import skew as sp_skew, kurtosis as sp_kurt
    skew_vals = np.zeros((n_obs, N_T))
    kurt_vals = np.zeros((n_obs, N_T))
    for i in range(n_obs):
        n = lengths[i]
        block = padded[i, :n, :]
        if n > 2:
            skew_vals[i] = sp_skew(block, axis=0, bias=False)
        if n > 3:
            kurt_vals[i] = sp_kurt(block, axis=0, bias=False)

    skew_df = pd.DataFrame(skew_vals, index=obs_idx, columns=[f"{c}_skew" for c in T_COLS])
    kurt_df = pd.DataFrame(kurt_vals, index=obs_idx, columns=[f"{c}_kurtosis" for c in T_COLS])

    delta_vals = np.zeros((n_obs, N_T))
    range_vals = np.zeros((n_obs, N_T))
    slope_vals = np.zeros((n_obs, N_T))
    slope_early = np.zeros((n_obs, N_T))
    slope_late = np.zeros((n_obs, N_T))

    for i in range(n_obs):
        n = lengths[i]
        block = padded[i, :n, :]
        delta_vals[i] = block[-1] - block[0]
        range_vals[i] = np.nanmax(block, axis=0) - np.nanmin(block, axis=0)
        if n > 1:
            x = np.arange(n, dtype=float)
            xm = x.mean()
            xv = np.sum((x - xm) ** 2)
            for j in range(N_T):
                slope_vals[i, j] = np.sum((x - xm) * (block[:, j] - block[:, j].mean())) / xv
        half = n // 2
        if n >= 4:
            xe = np.arange(half, dtype=float)
            if len(xe) >= 2:
                xem, xev = xe.mean(), np.sum((xe - xe.mean()) ** 2)
                for j in range(N_T):
                    seg = block[:half, j]
                    slope_early[i, j] = np.sum((xe - xem) * (seg - seg.mean())) / xev
            xl = np.arange(n - half, dtype=float)
            if len(xl) >= 2:
                xlm, xlv = xl.mean(), np.sum((xl - xl.mean()) ** 2)
                for j in range(N_T):
                    seg = block[half:n, j]
                    slope_late[i, j] = np.sum((xl - xlm) * (seg - seg.mean())) / xlv

    trend_df = pd.DataFrame(
        np.hstack([delta_vals, range_vals, slope_vals, slope_early, slope_late]),
        index=obs_idx,
        columns=(
            [f"{c}_delta" for c in T_COLS] + [f"{c}_range" for c in T_COLS]
            + [f"{c}_slope" for c in T_COLS] + [f"{c}_slope_early" for c in T_COLS]
            + [f"{c}_slope_late" for c in T_COLS]
        ),
    )

    early_means = np.zeros((n_obs, N_T))
    late_means = np.zeros((n_obs, N_T))
    last5_means = np.zeros((n_obs, N_T))
    for i in range(n_obs):
        n = lengths[i]
        half = n // 2
        early_means[i] = np.nanmean(padded[i, :half, :], axis=0)
        late_means[i] = np.nanmean(padded[i, half:n, :], axis=0)
        last5_means[i] = np.nanmean(padded[i, max(0, n - 5):n, :], axis=0)

    seg_df = pd.DataFrame(
        np.hstack([early_means, late_means, last5_means]),
        index=obs_idx,
        columns=(
            [f"{c}_early_mean" for c in T_COLS] + [f"{c}_late_mean" for c in T_COLS]
            + [f"{c}_last5_mean" for c in T_COLS]
        ),
    )

    volatility = np.zeros((n_obs, N_T))
    autocorr1 = np.zeros((n_obs, N_T))
    for i in range(n_obs):
        n = lengths[i]
        block = padded[i, :n, :]
        if n > 1:
            volatility[i] = np.mean(np.abs(np.diff(block, axis=0)), axis=0)
        if n >= 3:
            for j in range(N_T):
                s = block[:, j]
                if np.var(s) > 0:
                    autocorr1[i, j] = np.corrcoef(s[:-1], s[1:])[0, 1]

    dyn_df = pd.DataFrame(
        np.hstack([volatility, np.nan_to_num(autocorr1, 0.0)]),
        index=obs_idx,
        columns=[f"{c}_volatility" for c in T_COLS] + [f"{c}_autocorr1" for c in T_COLS],
    )

    fft_mag1 = np.zeros((n_obs, N_T))
    fft_mag2 = np.zeros((n_obs, N_T))
    fft_freq1 = np.zeros((n_obs, N_T))
    fft_freq2 = np.zeros((n_obs, N_T))
    for i in range(n_obs):
        n = lengths[i]
        if n >= 4:
            for j in range(N_T):
                s = padded[i, :n, j]
                s_detrended = s - np.linspace(s[0], s[-1], n)
                fft_vals = np.abs(np.fft.rfft(s_detrended))[1:]
                freqs = np.fft.rfftfreq(n, d=1.0)[1:]
                if len(fft_vals) >= 2:
                    top2 = np.argsort(fft_vals)[-2:]
                    fft_mag1[i, j] = fft_vals[top2[-1]]
                    fft_freq1[i, j] = freqs[top2[-1]]
                    fft_mag2[i, j] = fft_vals[top2[-2]]
                    fft_freq2[i, j] = freqs[top2[-2]]
                elif len(fft_vals) == 1:
                    fft_mag1[i, j] = fft_vals[0]
                    fft_freq1[i, j] = freqs[0]

    fft_df = pd.DataFrame(
        np.hstack([fft_mag1, fft_mag2, fft_freq1, fft_freq2]),
        index=obs_idx,
        columns=(
            [f"{c}_fft_mag1" for c in T_COLS] + [f"{c}_fft_mag2" for c in T_COLS]
            + [f"{c}_fft_freq1" for c in T_COLS] + [f"{c}_fft_freq2" for c in T_COLS]
        ),
    )

    energy = np.zeros((n_obs, N_T))
    rms = np.zeros((n_obs, N_T))
    for i in range(n_obs):
        n = lengths[i]
        block = padded[i, :n, :]
        energy[i] = np.sum(block ** 2, axis=0)
        rms[i] = np.sqrt(np.mean(block ** 2, axis=0))

    energy_df = pd.DataFrame(
        np.hstack([energy, rms]),
        index=obs_idx,
        columns=[f"{c}_energy" for c in T_COLS] + [f"{c}_rms" for c in T_COLS],
    )

    obs_length = obs_size.rename("obs_length")

    result = pd.concat(
        [static, basic, *quants.values(), iqr_df, skew_df, kurt_df,
         trend_df, seg_df, dyn_df, fft_df, energy_df, obs_length],
        axis=1,
    ).reset_index()

    from itertools import combinations
    for c1, c2 in combinations(T_COLS, 2):
        result[f"{c1}_{c2}_mean_diff"] = result[f"{c1}_mean"] - result[f"{c2}_mean"]
        denom = result[f"{c2}_mean"].replace(0, np.nan)
        result[f"{c1}_{c2}_mean_ratio"] = result[f"{c1}_mean"] / denom

    t_pairs = list(combinations(range(N_T), 2))
    corr_data = np.zeros((n_obs, len(t_pairs)))
    for idx, (j1, j2) in enumerate(t_pairs):
        for i in range(n_obs):
            n = lengths[i]
            if n > 2:
                s1, s2 = padded[i, :n, j1], padded[i, :n, j2]
                if s1.std() > 0 and s2.std() > 0:
                    corr_data[i, idx] = np.corrcoef(s1, s2)[0, 1]

    corr_cols = [f"{T_COLS[j1]}_{T_COLS[j2]}_corr" for j1, j2 in t_pairs]
    for ci, col_name in enumerate(corr_cols):
        result[col_name] = corr_data[:, ci]

    result = result.fillna(0.0)

    return result


print("Collapsing observations...")
X_train_df = collapse_obs(df_train)
X_test_df = collapse_obs(df_test)

y_train = df_train.groupby("obs")[["y_1", "y_2"]].first().reset_index(drop=True)
obs_to_patient = df_train.groupby("obs")["sub_id"].first().reset_index(drop=True)

test_obs = X_test_df["obs"].copy()
X_train_all = X_train_df.drop(columns=["obs"])
X_test_all = X_test_df.drop(columns=["obs"])

assert list(X_train_all.columns) == list(X_test_all.columns), "Column mismatch!"
assert X_train_all.isna().sum().sum() == 0, "Train has NaNs!"
assert X_test_all.isna().sum().sum() == 0, "Test has NaNs!"

CAT_FEATURE_NAMES = CAT_COLS

print(f"Train: {X_train_all.shape[0]} obs x {X_train_all.shape[1]} features")
print(f"Test:  {X_test_all.shape[0]} obs x {X_test_all.shape[1]} features")
print(f"Unique patients: {obs_to_patient.nunique()}")

## 3. Feature Selection

189 features for 14k observations is prone to overfitting. We train a quick LightGBM per target, rank features by importance (gain), and keep the top 100 per target (union). Static features (num_*, cat_*) are always kept.

In [ ]:
TOP_K = 100

important_features = set()
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

for i, target in enumerate(["y_1", "y_2"]):
    y_target = y_train[target]

    model = lgb.LGBMRegressor(
        objective="regression", n_estimators=500, learning_rate=0.03,
        max_depth=5, num_leaves=31, subsample=0.7, colsample_bytree=0.6,
        verbosity=-1, random_state=SEED,
    )
    model.fit(X_train_all, y_target)

    importance = pd.Series(
        model.feature_importances_, index=X_train_all.columns
    ).sort_values(ascending=False)

    top_feats = importance.head(TOP_K).index.tolist()
    important_features.update(top_feats)

    importance.head(30).plot.barh(ax=axes[i], color="steelblue")
    axes[i].set_title(f"Top 30 Features — {target}")
    axes[i].invert_yaxis()
    axes[i].set_xlabel("Importance (gain)")

plt.tight_layout()
plt.show()

important_features.update(NUM_COLS + CAT_COLS)
selected_features = sorted(important_features)

print(f"\nFeature selection: {X_train_all.shape[1]} → {len(selected_features)} features")
print(f"  Dropped: {X_train_all.shape[1] - len(selected_features)} noise features")

X_train_sel = X_train_all[selected_features].copy()
X_test_sel = X_test_all[selected_features].copy()

CAT_FEATURE_NAMES = [c for c in CAT_COLS if c in selected_features]

print(f"  Final train: {X_train_sel.shape}")
print(f"  Final test:  {X_test_sel.shape}")

## 4. Bayesian Hyperparameter Tuning (Optuna)

Tuning all 3 models per target. MSE loss with MAE eval — MSE gives smoother gradients for better convergence. y_1 is trained in log1p space (right-skewed), y_2 stays raw (symmetric).

In [ ]:
TUNE_FOLDS = 3
N_TRIALS_LGBM = 30
N_TRIALS_CB = 20
N_TRIALS_XGB = 30

USE_LOG1P = {"y_1": True, "y_2": False}


def transform_target(y, target):
    if USE_LOG1P.get(target, False):
        return np.log1p(y)
    return y


def inverse_transform_target(y, target):
    if USE_LOG1P.get(target, False):
        return np.expm1(y)
    return y


def objective_lgbm(trial, X, y, groups, target):
    params = {
        "objective": "regression",
        "metric": "mae",
        "verbosity": -1,
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.008, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 40),
        "subsample": trial.suggest_float("subsample", 0.5, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.8),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 5.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.5),
        "random_state": SEED,
    }

    y_t = transform_target(y, target)
    gkf = GroupKFold(n_splits=TUNE_FOLDS)
    maes = []
    for tr_idx, val_idx in gkf.split(X, y_t, groups=groups):
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X.iloc[tr_idx], y_t.iloc[tr_idx],
            eval_set=[(X.iloc[val_idx], y_t.iloc[val_idx])],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
        )
        preds = inverse_transform_target(model.predict(X.iloc[val_idx]), target)
        maes.append(mean_absolute_error(y.iloc[val_idx], preds))

    return np.mean(maes)


def objective_xgb(trial, X, y, groups, target):
    params = {
        "n_estimators": 1000,
        "tree_method": "hist",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 15),
        "subsample": trial.suggest_float("subsample", 0.5, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.8),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 5.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "early_stopping_rounds": 50,
        "random_state": SEED,
    }

    y_t = transform_target(y, target)
    gkf = GroupKFold(n_splits=TUNE_FOLDS)
    maes = []
    for tr_idx, val_idx in gkf.split(X, y_t, groups=groups):
        model = xgb.XGBRegressor(**params)
        model.fit(X.iloc[tr_idx], y_t.iloc[tr_idx],
                  eval_set=[(X.iloc[val_idx], y_t.iloc[val_idx])], verbose=False)
        preds = inverse_transform_target(model.predict(X.iloc[val_idx]), target)
        maes.append(mean_absolute_error(y.iloc[val_idx], preds))

    return np.mean(maes)


def objective_catboost(trial, X, y, groups, target):
    cat_indices = [X.columns.get_loc(c) for c in CAT_FEATURE_NAMES]
    params = {
        "iterations": 1000,
        "loss_function": "RMSE",
        "eval_metric": "MAE",
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.08, log=True),
        "depth": trial.suggest_int("depth", 4, 7),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.5, 5.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 0.9),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 35),
        "random_strength": trial.suggest_float("random_strength", 0.5, 5.0, log=True),
        "cat_features": cat_indices,
        "random_seed": SEED,
        "verbose": 0,
        "early_stopping_rounds": 50,
        "train_dir": "/tmp/catboost_info",
    }

    y_t = transform_target(y, target)
    gkf = GroupKFold(n_splits=TUNE_FOLDS)
    maes = []
    for tr_idx, val_idx in gkf.split(X, y_t, groups=groups):
        model = CatBoostRegressor(**params)
        model.fit(
            X.iloc[tr_idx], y_t.iloc[tr_idx],
            eval_set=(X.iloc[val_idx], y_t.iloc[val_idx]),
            verbose=0,
        )
        preds = inverse_transform_target(model.predict(X.iloc[val_idx]), target)
        maes.append(mean_absolute_error(y.iloc[val_idx], preds))

    return np.mean(maes)


best_params = {"lgbm": {}, "xgb": {}, "catboost": {}}

for target in ["y_1", "y_2"]:
    y_target = y_train[target]
    log_str = " (log1p space)" if USE_LOG1P[target] else ""

    print(f"\nTuning LightGBM for {target}{log_str} ({N_TRIALS_LGBM} trials)...")
    study_lgbm = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study_lgbm.optimize(lambda trial: objective_lgbm(trial, X_train_sel, y_target, obs_to_patient, target), n_trials=N_TRIALS_LGBM)
    best_params["lgbm"][target] = study_lgbm.best_params
    print(f"  Best MAE: {study_lgbm.best_value:.4f}")
    print(f"  Best params: {study_lgbm.best_params}")

    print(f"\nTuning XGBoost for {target}{log_str} ({N_TRIALS_XGB} trials)...")
    study_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study_xgb.optimize(lambda trial: objective_xgb(trial, X_train_sel, y_target, obs_to_patient, target), n_trials=N_TRIALS_XGB)
    best_params["xgb"][target] = study_xgb.best_params
    print(f"  Best MAE: {study_xgb.best_value:.4f}")
    print(f"  Best params: {study_xgb.best_params}")

    print(f"\nTuning CatBoost for {target}{log_str} ({N_TRIALS_CB} trials)...")
    study_cb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study_cb.optimize(lambda trial: objective_catboost(trial, X_train_sel, y_target, obs_to_patient, target), n_trials=N_TRIALS_CB)
    best_params["catboost"][target] = study_cb.best_params
    print(f"  Best MAE: {study_cb.best_value:.4f}")
    print(f"  Best params: {study_cb.best_params}")

print("\n" + "="*60)
print("Tuning complete!")
print("="*60)

## 5. Train Final Models + OOF Predictions

Using the tuned params from Optuna. Each model is trained with 3 different seeds and averaged to reduce variance. 5-fold GroupKFold OOF predictions become the meta-features for stacking.

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)

ALL_MODELS = ["lgbm", "xgb", "catboost"]

oof_preds = {t: {} for t in ["y_1", "y_2"]}
test_preds = {t: {} for t in ["y_1", "y_2"]}

cat_indices = [X_train_sel.columns.get_loc(c) for c in CAT_FEATURE_NAMES]

for target in ["y_1", "y_2"]:
    y_target = y_train[target]
    y_target_t = transform_target(y_target, target)
    log_str = " (log1p)" if USE_LOG1P[target] else ""

    print(f"\n{'='*50}")
    print(f"Training for {target}{log_str}")
    print(f"{'='*50}")

    for model_name in ALL_MODELS:
        oof_seed_sum = np.zeros(len(X_train_sel))
        test_seed_sum = np.zeros(len(X_test_sel))

        for seed in SEEDS:
            oof = np.zeros(len(X_train_sel))
            test_fold_preds = []
            bp = best_params[model_name][target]

            for fold_i, (tr_idx, val_idx) in enumerate(gkf.split(X_train_sel, y_target_t, groups=obs_to_patient)):
                X_tr = X_train_sel.iloc[tr_idx]
                y_tr = y_target_t.iloc[tr_idx]
                X_val = X_train_sel.iloc[val_idx]
                y_val = y_target_t.iloc[val_idx]

                if model_name == "lgbm":
                    model = lgb.LGBMRegressor(
                        objective="regression", metric="mae",
                        n_estimators=1000, verbosity=-1, random_state=seed,
                        **bp,
                    )
                    model.fit(
                        X_tr, y_tr,
                        eval_set=[(X_val, y_val)],
                        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
                    )
                elif model_name == "xgb":
                    model = xgb.XGBRegressor(
                        n_estimators=1000, tree_method="hist",
                        early_stopping_rounds=50, random_state=seed,
                        **bp,
                    )
                    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
                else:
                    model = CatBoostRegressor(
                        iterations=1000, loss_function="RMSE", eval_metric="MAE",
                        cat_features=cat_indices,
                        random_seed=seed, verbose=0, early_stopping_rounds=50,
                        train_dir="/tmp/catboost_info",
                        **bp,
                    )
                    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)

                oof[val_idx] = model.predict(X_val)
                test_fold_preds.append(model.predict(X_test_sel))

            oof_seed_sum += oof
            test_seed_sum += np.mean(test_fold_preds, axis=0)

        oof_avg_t = oof_seed_sum / len(SEEDS)
        test_avg_t = test_seed_sum / len(SEEDS)

        oof_preds[target][model_name] = inverse_transform_target(oof_avg_t, target)
        test_preds[target][model_name] = inverse_transform_target(test_avg_t, target)

        oof_mae = mean_absolute_error(y_target, oof_preds[target][model_name])
        print(f"  {model_name} {target} OOF MAE (seed-avg): {oof_mae:.4f}")

## 6. Meta-Learner (Stacking)

Stack the 3 base model OOF predictions with Ridge or Lasso, selected via CV. Operates in original scale (OOF predictions are already inverse-transformed).

In [ ]:
MODEL_NAMES = ALL_MODELS
meta_models = {}

for target in ["y_1", "y_2"]:
    y_target = y_train[target].values
    S_train = np.column_stack([oof_preds[target][m] for m in MODEL_NAMES])

    candidates = {}
    for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
        candidates[f"Ridge({alpha})"] = Ridge(alpha=alpha)
    for alpha in [0.001, 0.01, 0.1]:
        candidates[f"Lasso({alpha})"] = Lasso(alpha=alpha, max_iter=5000)

    best_score = float("inf")
    best_name = None

    meta_gkf = GroupKFold(n_splits=3)
    for name, meta in candidates.items():
        scores = []
        for tr_idx, val_idx in meta_gkf.split(S_train, y_target, groups=obs_to_patient):
            m = type(meta)(**meta.get_params())
            m.fit(S_train[tr_idx], y_target[tr_idx])
            preds = m.predict(S_train[val_idx])
            scores.append(mean_absolute_error(y_target[val_idx], preds))
        mean_score = np.mean(scores)
        if mean_score < best_score:
            best_score = mean_score
            best_name = name

    best_meta = type(candidates[best_name])(**candidates[best_name].get_params())
    best_meta.fit(S_train, y_target)
    meta_models[target] = best_meta

    print(f"{target}: best meta = {best_name} (CV MAE: {best_score:.4f})")
    if hasattr(best_meta, "coef_"):
        weights = dict(zip(MODEL_NAMES, best_meta.coef_.round(4)))
        print(f"  Weights: {weights}, intercept: {best_meta.intercept_:.4f}")

    avg_mae = mean_absolute_error(y_target, S_train.mean(axis=1))
    print(f"  Simple avg OOF MAE: {avg_mae:.4f}")
    print(f"  Stacked OOF MAE:    {best_score:.4f}")

## 7. Outer-CV Evaluation

Honest comparison across all methods using the same GroupKFold splits.

In [17]:
outer_gkf = GroupKFold(n_splits=N_FOLDS)

outer_scores = {t: {m: [] for m in MODEL_NAMES + ["avg", "stacked"]} for t in ["y_1", "y_2"]}

for fold_i, (tr_idx, val_idx) in enumerate(outer_gkf.split(X_train_sel, y_train["y_1"], groups=obs_to_patient)):
    print(f"\nOuter fold {fold_i+1}/{N_FOLDS} (train={len(tr_idx)}, val={len(val_idx)})")

    for target in ["y_1", "y_2"]:
        y_val = y_train[target].iloc[val_idx].values

        base_preds = {}
        for model_name in MODEL_NAMES:
            preds = oof_preds[target][model_name][val_idx]
            base_preds[model_name] = preds
            mae = mean_absolute_error(y_val, preds)
            outer_scores[target][model_name].append(mae)

        S_val = np.column_stack([base_preds[m] for m in MODEL_NAMES])

        avg_preds = S_val.mean(axis=1)
        outer_scores[target]["avg"].append(mean_absolute_error(y_val, avg_preds))

        stacked_preds = meta_models[target].predict(S_val)
        outer_scores[target]["stacked"].append(mean_absolute_error(y_val, stacked_preds))

# Summary table
print("\n" + "="*65)
print("   OUTER-CV MAE (GroupKFold by patient, 5 folds)")
print("="*65)

DISPLAY = {"lgbm": "LightGBM", "catboost": "CatBoost", "xgb": "XGBoost",
           "avg": "Simple Avg", "stacked": "STACKED"}
rows = []
for method in MODEL_NAMES + ["avg", "stacked"]:
    y1_mae = np.mean(outer_scores["y_1"][method])
    y2_mae = np.mean(outer_scores["y_2"][method])
    rows.append({"Model": DISPLAY[method], "y_1 MAE": y1_mae, "y_2 MAE": y2_mae, "Mean MAE": (y1_mae + y2_mae) / 2})

results_df = pd.DataFrame(rows).sort_values("Mean MAE")
print(results_df.to_string(index=False, float_format="{:.4f}".format))

print("\n" + "-"*65)
print("Previous Kaggle score (v3):    4.3148")
print("v1 baseline (james_stack v1):  4.1685")
v4_best = results_df["Mean MAE"].min()
print(f"v4 best (this notebook):       {v4_best:.4f}")
delta = 4.3148 - v4_best
print(f"Expected improvement over previous Kaggle: {delta:.4f} ({delta/4.3148*100:.2f}%)")
print("-"*65)


Outer fold 1/5 (train=11536, val=2884)

Outer fold 2/5 (train=11536, val=2884)

Outer fold 3/5 (train=11536, val=2884)

Outer fold 4/5 (train=11536, val=2884)

Outer fold 5/5 (train=11536, val=2884)

   OUTER-CV MAE (GroupKFold by patient, 5 folds)
     Model  y_1 MAE  y_2 MAE  Mean MAE
   STACKED   4.9721   3.3535    4.1628
Simple Avg   4.9648   3.3743    4.1696
  CatBoost   4.9700   3.3708    4.1704
   XGBoost   4.9779   3.3928    4.1854
  LightGBM   4.9811   3.3917    4.1864

-----------------------------------------------------------------
Previous Kaggle score (v3):    4.3148
v1 baseline (james_stack v1):  4.1685
v4 best (this notebook):       4.1628
Expected improvement over previous Kaggle: 0.1520 (3.52%)
-----------------------------------------------------------------


## 8. Submission

In [18]:
submission = pd.DataFrame({"obs": test_obs})

for target in ["y_1", "y_2"]:
    S_test = np.column_stack([test_preds[target][m] for m in MODEL_NAMES])
    submission[target] = meta_models[target].predict(S_test)

# Sanity checks
for target in ["y_1", "y_2"]:
    train_vals = y_train[target]
    pred_vals = submission[target]
    print(f"{target}:")
    print(f"  Train — mean: {train_vals.mean():.2f}, std: {train_vals.std():.2f}, range: [{train_vals.min():.1f}, {train_vals.max():.1f}]")
    print(f"  Preds — mean: {pred_vals.mean():.2f}, std: {pred_vals.std():.2f}, range: [{pred_vals.min():.1f}, {pred_vals.max():.1f}]")

submission.to_csv("james_v4_submission.csv", index=False)
print(f"\nSaved: james_v4_submission.csv ({submission.shape[0]} rows)")
submission.head(10)

y_1:
  Train — mean: 44.50, std: 14.31, range: [0.5, 134.5]
  Preds — mean: 42.30, std: 11.72, range: [9.2, 84.2]
y_2:
  Train — mean: 83.51, std: 16.23, range: [0.0, 190.0]
  Preds — mean: 82.54, std: 15.28, range: [48.0, 144.6]

Saved: james_v4_submission.csv (3450 rows)


,obs,y_1,y_2
0,18,41.043923,104.912416
1,19,34.155041,101.144120
2,20,36.264099,96.608711
3,21,35.197248,95.256102
4,22,35.522125,94.571300
5,23,32.684435,57.245845
6,24,43.118251,57.837541
7,25,35.347122,54.189904
8,26,38.438137,48.682639
9,27,34.322768,48.803387
